# Notebook 3 — Modelo semântico, análise de resultados e `results.txt`
**Pessoa 3 (Mariana Almeida).** Parte das entregas dos notebooks 1 e 2.

Resultado: tabela final com intervalos, análise de erros por especialidade, exemplos de previsões certas e erradas e o `results.txt`.

## 1. Imports e dados

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

#Same paths as notebook 1: notebooks/ sits next to data/ and results/
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_folder = root / 'data'
results_folder = root / 'results'

specialties = ['Cardiovascular-Pulmonary', 'Dermatology', 'Gastroenterology','General Medicine', 'Neurology', 'Neurosurgery', 'Obstetrics-Gynecology','Ophthalmology', 'Orthopedic', 'Psychiatry-Psychology', 'Radiology', 'Surgery',]

#Folds saved by notebook 1 (same grouped splits for every model, so results are comparable)
splits = json.loads((results_folder / 'notebook1' / 'splits_seed42.json').read_text(encoding='utf-8'))['folds']

study_folder = results_folder / 'notebook3'
study_folder.mkdir(parents=True, exist_ok=True)

## 2. Modelo semântico pré-treinado
Opções: embeddings de frases (p.ex. `all-MiniLM-L6-v2`) + regressão logística (rápido, corre no CPU), ou afinar Bio_ClinicalBERT (`emilyalsentzer/Bio_ClinicalBERT`, precisa de GPU/Colab).
Avaliar **nas mesmas divisões** do notebook 1.

In [ ]:
# TODO

## 3. Tabela com as três partes + intervalos
Juntar as métricas dos notebooks 1, 2 e 3. Intervalos de confiança a 95% por bootstrap sobre as previsões fora da divisão.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

#Bootstrap 95% interval for accuracy, macro-F1 and min per-class F1
def bootstrap_interval(gold, pred, n=1000, seed=42):
    gold, pred = np.asarray(gold), np.asarray(pred)
    rng = np.random.default_rng(seed)
    scores = {'accuracy': [], 'macro_f1': [], 'min_f1': []}
    for _ in range(n):
        idx = rng.integers(0, len(gold), len(gold))
        g, p = gold[idx], pred[idx]
        f1s = f1_score(g, p, labels=specialties, average=None, zero_division=0)
        scores['accuracy'].append(accuracy_score(g, p))
        scores['macro_f1'].append(f1s.mean())
        scores['min_f1'].append(f1s.min())
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in scores.items()}

## 4. Onde é que os modelos falham: especialidades e tipos de exemplos

In [ ]:
# TODO: per-class F1, confusion matrix with readable labels, errors on duplicated texts vs unique texts

## 5. Exemplos de previsões certas e erradas
Pelo menos 3 erros concretos para a Discussão: excerto do texto, etiqueta prevista, etiqueta correta e interpretação.

In [ ]:
# TODO

## 6. `results.txt`
Usa o modelo final do notebook 2 no **teste limpo** (`data/test_no_labels.csv`, 398 casos).
O teste só serve para isto: não entra em treino nem no vocabulário do TF-IDF.

In [ ]:
#Writes one label per line, no header, same order as the test file, and checks the result
def write_results(predictions, n_test, path=root / 'results.txt'):
    predictions = [str(p).strip() for p in predictions]
    assert len(predictions) == n_test, f'{len(predictions)} predictions for {n_test} test cases'
    invalid = sorted(set(predictions) - set(specialties))
    assert not invalid, f'invalid labels: {invalid}'
    Path(path).write_text('\n'.join(predictions) + '\n', encoding='utf-8')
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    assert len(lines) == n_test and lines == predictions
    print(f'{path}: {len(lines)} lines OK')
    print(pd.Series(lines).value_counts())

In [ ]:
# test = pd.read_csv(data_folder / 'test_no_labels.csv', sep=';', names=['description', 'sample_name', 'transcription', 'keywords'], keep_default_na=False)
# model = ...  # final model from notebook 2
# write_results(model.predict(test), len(test))

## 7. Notas para o Gonçalo (relatório)
- Tabelas e figuras a usar:
- Exemplos escolhidos:
- Limitações: